In [1]:
import os
import torch

# GPU check
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Dataset path check
dataset_path = "/kaggle/input/datasets/faridrash/rgbd-dataset-freiburg1-desk"
print("\nContents of dataset root:")
print(os.listdir(dataset_path))

Torch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4

Contents of dataset root:
['rgbd_dataset_freiburg1_desk.tgz']


In [2]:
import tarfile

archive_path = "/kaggle/input/datasets/faridrash/rgbd-dataset-freiburg1-desk/rgbd_dataset_freiburg1_desk.tgz"
extract_dir = "/kaggle/working/data"

os.makedirs(extract_dir, exist_ok=True)

with tarfile.open(archive_path, "r:gz") as tar:
    tar.extractall(path=extract_dir)

print("Extracted. Contents of extract_dir:")
print(os.listdir(extract_dir))

/tmp/ipykernel_58/2503886923.py:9: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_dir)


Extracted. Contents of extract_dir:
['rgbd_dataset_freiburg1_desk']


In [3]:
data_root = "/kaggle/working/data/rgbd_dataset_freiburg1_desk"

print("Contents:", os.listdir(data_root))

rgb_txt_path = os.path.join(data_root, "rgb.txt")
with open(rgb_txt_path) as f:
    lines = [l for l in f if not l.startswith("#")]

print(f"\nrgb.txt line count (excluding comments): {len(lines)}")
print("First line:", lines[0].strip())
print("Last line:", lines[-1].strip())

Contents: ['accelerometer.txt', 'groundtruth.txt', 'depth.txt', 'rgb', 'depth', 'rgb.txt']

rgb.txt line count (excluding comments): 613
First line: 1305031452.791720 rgb/1305031452.791720.png
Last line: 1305031473.196069 rgb/1305031473.196069.png


In [4]:
import json

# Load poses.json (this is the source of truth for which frames SLAM used)
poses_path = "/kaggle/input/datasets/faridrash/posses/poses.json"
with open(poses_path) as f:
    poses = json.load(f)

print("Poses count:", len(poses))

# Extract the exact timestamps SLAM used, formatted to match TUM's 6-decimal filename convention
slam_timestamps = set(f"{p['timestamp']:.6f}" for p in poses)
print("Unique SLAM timestamps:", len(slam_timestamps))
print("Sample:", list(slam_timestamps)[:3])

Poses count: 571
Unique SLAM timestamps: 571
Sample: ['1305031462.492008', '1305031466.495809', '1305031463.995821']


In [5]:
data_root = "/kaggle/working/data/rgbd_dataset_freiburg1_desk"
rgb_txt_path = os.path.join(data_root, "rgb.txt")

with open(rgb_txt_path) as f:
    rgb_lines = [l.strip() for l in f if not l.startswith("#")]

# Each line looks like: "1305031452.791720 rgb/1305031452.791720.png"
rgb_entries = []
for line in rgb_lines:
    ts_str, rel_path = line.split()
    rgb_entries.append((ts_str, rel_path))

print("Total raw rgb.txt entries:", len(rgb_entries))

# Filter to only the timestamps SLAM actually used
matched = [(ts, path) for ts, path in rgb_entries if ts in slam_timestamps]

print("Matched entries:", len(matched))

# Check for any SLAM timestamps that did NOT find a match (should be empty)
matched_ts_set = set(ts for ts, _ in matched)
unmatched_slam = slam_timestamps - matched_ts_set
print("SLAM timestamps with no rgb.txt match:", len(unmatched_slam))
if unmatched_slam:
    print("Examples:", list(unmatched_slam)[:5])

Total raw rgb.txt entries: 613
Matched entries: 571
SLAM timestamps with no rgb.txt match: 0


In [6]:
timestamps_only = [float(ts) for ts, _ in matched]
is_sorted = timestamps_only == sorted(timestamps_only)
print("Matched list is chronologically sorted:", is_sorted)

print("\nFirst 3 matched entries:")
for ts, path in matched[:3]:
    print(ts, path)

print("\nLast 3 matched entries:")
for ts, path in matched[-3:]:
    print(ts, path)

Matched list is chronologically sorted: True

First 3 matched entries:
1305031453.423683 rgb/1305031453.423683.png
1305031453.459685 rgb/1305031453.459685.png
1305031453.491698 rgb/1305031453.491698.png

Last 3 matched entries:
1305031473.127744 rgb/1305031473.127744.png
1305031473.167060 rgb/1305031473.167060.png
1305031473.196069 rgb/1305031473.196069.png


In [7]:
pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 29.3 MB/s eta 0:00:0000:01
Note: you may need to restart the kernel to use updated packages.


In [8]:
from ultralytics import YOLO
import ultralytics, os

print("Ultralytics location:", os.path.dirname(ultralytics.__file__))

# Find bundled tracker configs
tracker_cfg_dir = os.path.join(os.path.dirname(ultralytics.__file__), "cfg", "trackers")
print("\nTracker configs available:")
print(os.listdir(tracker_cfg_dir))

# Inspect the track() method signature
help(YOLO.track)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics location: /usr/local/lib/python3.12/dist-packages/ultralytics

Tracker configs available:
['bytetrack.yaml', 'tracktrack.yaml', 'fasttrack.yaml', 'deepocsort.yaml', 'ocsort.yaml', 'botsort.yaml']
Help on function track in module ultralytics.engine.model:

track(self, source: 'str | Path | int | list | tuple | np.ndarray | torch.Tensor' = None, stream: 'bool' = False, persist: 'bool' = False, **kwargs: 'Any') -> 'list[Results]'
    Conduct object tracking on the specified input source using the registered trackers.

    This method performs object tracking using the model's predictors and optionally registered trackers. It handles
    various input sources such as file 

In [9]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")  # auto-downloads COCO-pretrained weights on first use

# Test on the first 3 synced frames
test_frames = matched[:3]
test_paths = [os.path.join(data_root, rel_path) for _, rel_path in test_frames]

print("Testing on:")
for p in test_paths:
    print(" ", p)

results = model.track(source=test_paths, persist=True, tracker="bytetrack.yaml")

print("\nNumber of results:", len(results))
print("\nFirst result type:", type(results[0]))
print("Boxes attribute:", results[0].boxes)

Testing on:
  /kaggle/working/data/rgbd_dataset_freiburg1_desk/rgb/1305031453.423683.png
  /kaggle/working/data/rgbd_dataset_freiburg1_desk/rgb/1305031453.459685.png
  /kaggle/working/data/rgbd_dataset_freiburg1_desk/rgb/1305031453.491698.png
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 279ms
Prepared 1 package in 71ms
Installed 1 package in 5ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


0: 480x640 1 tv, 1 mouse, 1 keyboard, 1 book, 22.5ms
1: 480x640 1 tv, 1 mouse, 1 keyboard, 1 book, 22.5ms
2: 480x640 1 tv, 1 mouse, 1 keyboard, 1 book, 22.5ms
Speed: 6.4ms preprocess, 22.5ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)

Number of results: 3

First result type: <class 'ultralytics.engine.results.Results'>
Boxes attribute: ultralytics.engine.results.Boxes object

In [10]:
# First, confirm how class names are stored
print("Sample class mappings:")
for idx in [62, 64, 66, 73]:
    print(idx, "->", model.names[idx])

Sample class mappings:
62 -> tv
64 -> mouse
66 -> keyboard
73 -> book


In [11]:
def extract_frame_detections(result, timestamp):
    """Convert one YOLO+ByteTrack Results object into a clean dict."""
    boxes = result.boxes

    detections = []
    if boxes.id is not None:  # id can be None if tracking hasn't assigned any yet
        for i in range(len(boxes)):
            cls_idx = int(boxes.cls[i].item())
            detections.append({
                "track_id": int(boxes.id[i].item()),
                "class_name": model.names[cls_idx],
                "confidence": round(float(boxes.conf[i].item()), 4),
                "bbox_xyxy": [round(v, 2) for v in boxes.xyxy[i].tolist()]
            })

    return {
        "timestamp": timestamp,
        "detections": detections
    }

# Test on the same 3 frames
test_output = []
for i, (ts, rel_path) in enumerate(test_frames):
    frame_data = extract_frame_detections(results[i], ts)
    test_output.append(frame_data)

import json
print(json.dumps(test_output, indent=2))

[
  {
    "timestamp": "1305031453.423683",
    "detections": [
      {
        "track_id": 1,
        "class_name": "tv",
        "confidence": 0.8471,
        "bbox_xyxy": [
          0.0,
          1.04,
          231.73,
          235.55
        ]
      },
      {
        "track_id": 2,
        "class_name": "mouse",
        "confidence": 0.8035,
        "bbox_xyxy": [
          41.5,
          276.19,
          111.06,
          308.45
        ]
      },
      {
        "track_id": 3,
        "class_name": "keyboard",
        "confidence": 0.5655,
        "bbox_xyxy": [
          0.09,
          303.98,
          263.4,
          479.64
        ]
      },
      {
        "track_id": 4,
        "class_name": "book",
        "confidence": 0.3361,
        "bbox_xyxy": [
          105.19,
          232.88,
          371.58,
          400.2
        ]
      }
    ]
  },
  {
    "timestamp": "1305031453.459685",
    "detections": [
      {
        "track_id": 1,
        "class_name": "tv

In [12]:
all_paths = [os.path.join(data_root, rel_path) for _, rel_path in matched]
all_timestamps = [ts for ts, _ in matched]

print(f"Running detection+tracking on {len(all_paths)} frames...")

full_results = model.track(source=all_paths, persist=True, tracker="bytetrack.yaml", verbose=False)

print("Done. Results count:", len(full_results))

Running detection+tracking on 571 frames...
Done. Results count: 571


In [13]:
all_tracks = []
for i, ts in enumerate(all_timestamps):
    frame_data = extract_frame_detections(full_results[i], ts)
    all_tracks.append(frame_data)

output_path = "/kaggle/working/tracks.json"
with open(output_path, "w") as f:
    json.dump(all_tracks, f, indent=2)

print(f"Saved {len(all_tracks)} frames to {output_path}")

# Quick sanity checks
total_detections = sum(len(frame["detections"]) for frame in all_tracks)
unique_track_ids = set()
unique_classes = set()
for frame in all_tracks:
    for det in frame["detections"]:
        unique_track_ids.add(det["track_id"])
        unique_classes.add(det["class_name"])

print(f"Total detections across all frames: {total_detections}")
print(f"Unique track IDs used: {len(unique_track_ids)}")
print(f"Unique classes detected: {unique_classes}")

Saved 571 frames to /kaggle/working/tracks.json
Total detections across all frames: 2260
Unique track IDs used: 104
Unique classes detected: {'keyboard', 'handbag', 'laptop', 'remote', 'chair', 'person', 'mouse', 'tv', 'cup', 'dining table', 'couch', 'book'}


In [14]:
from collections import defaultdict

track_frame_counts = defaultdict(int)
track_classes = {}

for frame in all_tracks:
    for det in frame["detections"]:
        tid = det["track_id"]
        track_frame_counts[tid] += 1
        track_classes[tid] = det["class_name"]

# Sort by frame count, descending
sorted_tracks = sorted(track_frame_counts.items(), key=lambda x: -x[1])

print(f"{'Track ID':<10}{'Class':<15}{'Frame Count':<12}")
for tid, count in sorted_tracks[:20]:
    print(f"{tid:<10}{track_classes[tid]:<15}{count:<12}")

print("\n...")
print(f"\nTracks lasting only 1 frame: {sum(1 for c in track_frame_counts.values() if c == 1)}")
print(f"Tracks lasting 1-5 frames: {sum(1 for c in track_frame_counts.values() if c <= 5)}")
print(f"Tracks lasting 100+ frames: {sum(1 for c in track_frame_counts.values() if c >= 100)}")

Track ID  Class          Frame Count 
467       keyboard       169         
167       laptop         124         
426       laptop         112         
433       tv             112         
396       chair          108         
1         tv             90          
18        laptop         89          
631       book           86          
220       laptop         77          
454       book           77          
66        book           75          
651       book           66          
318       laptop         57          
657       book           53          
393       tv             47          
224       keyboard       44          
673       keyboard       35          
325       laptop         29          
536       tv             29          
34        chair          26          

...

Tracks lasting only 1 frame: 5
Tracks lasting 1-5 frames: 34
Tracks lasting 100+ frames: 5


In [15]:
tracker_cfg_path = os.path.join(tracker_cfg_dir, "bytetrack.yaml")
with open(tracker_cfg_path) as f:
    print(f.read())

# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license

# ByteTrack tracker defaults for mode="track"
# Docs: https://docs.ultralytics.com/modes/track

tracker_type: bytetrack # (str) Tracker backend: bytetrack
track_high_thresh: 0.25 # (float) First-stage match threshold; raise for cleaner tracks, lower to keep more
track_low_thresh: 0.1 # (float) Second-stage threshold for low-score matches; balances recovery vs drift
new_track_thresh: 0.25 # (float) Start a new track if no match ≥ this; higher reduces false tracks
track_buffer: 30 # (int) Frames to keep lost tracks alive; higher handles occlusion, increases ID switches risk
match_thresh: 0.8 # (float) Association similarity threshold (IoU/cost); tune with detector quality
fuse_score: True # (bool) Fuse detection score with motion/IoU for matching; stabilizes weak detections

